# Massachusetts PDF parsing walkthrough

Learning step only — no collector, no SQLite writes.

Official source: [Massachusetts Gaming Commission revenue reports](https://massgaming.com/regulations/revenue/).

## Category 1 vs Category 3

| Category | What it is | Example licensees in the monthly PDF |
| --- | --- | --- |
| **Category 1** | Retail sportsbooks at Massachusetts casinos | Encore Boston Harbor, MGM Springfield, Plainridge Park Casino |
| **Category 3** | Online/mobile sportsbook brands licensed in MA | DraftKings, FanDuel, BetMGM, Caesars Sportsbook, … |

We want **Category 3 only** for statewide online handle/revenue series. Retail Category 1 rows,
`Total Retail`, `Total Online`, and the overall `Total` are control totals — not operator observations.

In [1]:
import json
import re
import sys
from io import BytesIO
from pathlib import Path
from urllib.parse import urljoin

import pandas as pd
import pdfplumber
import requests
from bs4 import BeautifulSoup

ROOT = Path.cwd()
if not (ROOT / "src").exists() and (ROOT.parent / "src").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

from variant_gaming.common import parse_money, project_root, save_raw_bytes, sha256_bytes, utc_now

ROOT = project_root()
LANDING_URL = "https://massgaming.com/regulations/revenue/"
BROWSER_HEADERS = {"User-Agent": "researchOS-variant-gaming/1.0 (+official regulator downloads)"}
REPORT_LABEL = "July 2026 consolidated revenue report"
PDF_FILENAME = "MGC-Revenue-Report-July-2026.pdf"

In [2]:
def discover_consolidated_report_url(html: str) -> str:
    soup = BeautifulSoup(html, "html.parser")
    for anchor in soup.find_all("a", href=True):
        href = urljoin(LANDING_URL, anchor["href"].strip())
        if href.endswith(PDF_FILENAME):
            return href
    raise RuntimeError(f"Could not find {PDF_FILENAME} on landing page")


def find_saved_pdf(root: Path, filename: str) -> Path | None:
    matches = sorted((root / "data" / "raw" / "MA").rglob(f"*_{filename}"))
    return matches[-1] if matches else None


landing_resp = requests.get(LANDING_URL, headers=BROWSER_HEADERS, timeout=60)
landing_resp.raise_for_status()
source_url = discover_consolidated_report_url(landing_resp.text)
saved_path = find_saved_pdf(ROOT, PDF_FILENAME)
meta_path = saved_path.with_suffix(saved_path.suffix + ".meta.json") if saved_path else None

if saved_path is not None and meta_path is not None and meta_path.exists():
    pdf_bytes = saved_path.read_bytes()
    meta = json.loads(meta_path.read_text(encoding="utf-8"))
    retrieved_at = pd.Timestamp(meta["retrieved_at_utc"]).to_pydatetime()
    source_url = meta.get("source_url", source_url)
else:
    retrieved_at = utc_now()
    pdf_resp = requests.get(source_url, headers=BROWSER_HEADERS, timeout=60)
    pdf_resp.raise_for_status()
    pdf_bytes = pdf_resp.content
    if not pdf_bytes.startswith(b"%PDF"):
        raise ValueError("Downloaded bytes do not look like a PDF (missing %PDF header)")
    saved_path = save_raw_bytes(ROOT, "MA", pdf_bytes, PDF_FILENAME, retrieved_at=retrieved_at)
    meta_path = saved_path.with_suffix(saved_path.suffix + ".meta.json")
    meta = {
        "source_url": source_url,
        "retrieved_at_utc": retrieved_at.isoformat(),
        "sha256": sha256_bytes(pdf_bytes),
        "saved_path": str(saved_path.relative_to(ROOT)).replace("\\", "/"),
    }
    meta_path.write_text(json.dumps(meta, indent=2), encoding="utf-8")

content_hash = sha256_bytes(pdf_bytes)

print("selected PDF:", REPORT_LABEL)
print("source_url:", source_url)
print("saved_path:", saved_path.relative_to(ROOT))
print("sha256:", content_hash)
print("retrieved_at_utc:", retrieved_at.isoformat())

selected PDF: July 2026 consolidated revenue report
source_url: https://massgaming.com/wp-content/uploads/MGC-Revenue-Report-July-2026.pdf
saved_path: data\raw\MA\2026-09-03\ccac154845a1a2e63dac0a33a9d725edf6eeaa2f2037365c9175596c92891517_MGC-Revenue-Report-July-2026.pdf
sha256: ccac154845a1a2e63dac0a33a9d725edf6eeaa2f2037365c9175596c92891517
retrieved_at_utc: 2026-09-03T02:17:58.615654+00:00


In [3]:
RETAIL_OPERATORS = {
    "Encore Boston Harbor",
    "MGM Springfield",
    "Plainridge Park Casino",
}
SKIP_LABELS = {"Total Retail", "Total Online", "Total", "ONLINE LICENSEE", "RETAIL LICENSEE"}
MONEY_FIELD = (
    r"(?:"
    r"\(\$[\d,]+\.\d{2}\)"
    r"|-\$[\d,]+\.\d{2}"
    r"|\$-[\d,]+\.\d{2}"
    r"|\$[\d,]+\.\d{2}"
    r")"
)
MONEY_ROW_RE = re.compile(
    rf"^{MONEY_FIELD}\s+{MONEY_FIELD}\s+(?P<hold>[\d.]+%)\s+{MONEY_FIELD}\s+{MONEY_FIELD}$"
)


def money_fields_from_row(line: str) -> tuple[float, float, float, float] | None:
    match = MONEY_ROW_RE.match(line)
    if not match:
        return None
    tokens = re.findall(MONEY_FIELD, line)
    if len(tokens) != 4:
        return None
    return tuple(parse_money(token) for token in tokens)


def find_online_operator_page(pdf: pdfplumber.PDF) -> tuple[int, str, list]:
    """Return page index, text, and tables for the single-month ONLINE LICENSEE section."""
    for page_index, page in enumerate(pdf.pages):
        text = page.extract_text() or ""
        upper = text.upper()
        has_online_detail = any(line.strip().upper() == "ONLINE LICENSEE" for line in text.splitlines())
        if not has_online_detail:
            continue
        if "MONTH OVER MONTH" in upper or "YEAR OVER YEAR" in upper:
            continue
        return page_index, text, page.extract_tables() or []
    raise ValueError("No single-month ONLINE LICENSEE page found")


def parse_online_section(text: str, *, period_start: str) -> tuple[pd.DataFrame, dict]:
    lines = [line.strip() for line in text.splitlines() if line.strip()]
    in_online = False
    operators: list[dict] = []
    total_online: dict | None = None

    idx = 0
    while idx < len(lines):
        line = lines[idx]
        if line.upper() == "ONLINE LICENSEE":
            in_online = True
            idx += 1
            continue
        if not in_online:
            idx += 1
            continue
        if line.lower() == "total" and idx > 0:
            amounts = money_fields_from_row(lines[idx - 1])
            if amounts is not None:
                total_online = {
                    "wagers_settled": amounts[0],
                    "taxable_revenue": amounts[2],
                    "tax_collected": amounts[3],
                }
            break
        amounts = money_fields_from_row(line)
        if amounts is not None and idx + 1 < len(lines):
            operator = lines[idx + 1]
            if operator in RETAIL_OPERATORS or operator in SKIP_LABELS:
                idx += 1
                continue
            if operator.lower() == "total":
                idx += 1
                continue
            operators.append(
                {
                    "operator": operator,
                    "period_start": period_start,
                    "wagers_settled": amounts[0],
                    "taxable_revenue": amounts[2],
                    "tax_collected": amounts[3],
                }
            )
            idx += 2
            continue
        idx += 1

    if not operators or total_online is None:
        raise ValueError("Could not parse Category 3 operators or Total Online row")
    return pd.DataFrame(operators), total_online


def extracted_table_preview(tables: list) -> pd.DataFrame:
    blob = ""
    for row in tables[0] if tables else []:
        for cell in row:
            if cell and "ONLINE LICENSEE" in str(cell):
                blob = str(cell)
                break
    lines = [line.strip() for line in blob.splitlines() if line.strip()]
    preview_rows = []
    for idx, line in enumerate(lines):
        if money_fields_from_row(line) and idx + 1 < len(lines):
            preview_rows.append({"extracted_line": line, "label": lines[idx + 1]})
    return pd.DataFrame(preview_rows)


for token in ("-$1,234.56", "$-1,234.56", "($1,234.56)"):
    assert parse_money(token) == -1234.56, token
negative_row = "$100.00 $200.00 5.00% -$1,234.56 $50.00"
negative_amounts = money_fields_from_row(negative_row)
assert negative_amounts is not None
assert negative_amounts[2] == -1234.56
print("negative-value checks: passed")

with pdfplumber.open(BytesIO(pdf_bytes)) as pdf:
    page_index, page_text, tables = find_online_operator_page(pdf)

period_start = "2026-07-01"
category3, total_online = parse_online_section(page_text, period_start=period_start)
extracted_table = extracted_table_preview(tables)

print(f"relevant page: {page_index + 1}")
print("--- extracted page text (online section) ---")
start = page_text.upper().find("ONLINE LICENSEE")
print(page_text[start : start + 900])
extracted_table

negative-value checks: passed


relevant page: 3
--- extracted page text (online section) ---
ONLINE LICENSEE
Wagers Settled by Licensee Hold % Gaming Revenue (20% for Online/Mobile)
$3,272,045.97 $392,145.69 11.98% $383,965.58 $76,793.12
Bally Bet
$52,791,146.49 $6,518,544.14 12.35% $6,386,566.27 $1,277,313.25
BetMGM
$17,603,582.28 $1,173,632.04 6.67% $1,129,965.58 $225,993.01
Caesars Sportsbook
$281,176,332.44 $33,004,446.68 11.74% $32,324,402.68 $6,464,880.54
DraftKings
$59,682,063.71 $6,599,625.36 11.06% $6,453,759.09 $1,290,751.82
Fanatics
$150,594,846.88 $17,165,339.33 11.40% $16,801,318.96 $3,360,263.79
FanDuel
$21,981,883.07 $1,676,874.69 7.63% $1,622,011.39 $324,402.28
theScore Bet
$587,101,900.84 $66,530,607.93 11.33% $65,101,989.55 $13,020,397.81
Total


,extracted_line,label
0,"$3,272,045.97 $392,145.69 11.98% $383,965.58 $...",Bally Bet
1,"$52,791,146.49 $6,518,544.14 12.35% $6,386,566...",BetMGM
2,"$17,603,582.28 $1,173,632.04 6.67% $1,129,965....",Caesars Sportsbook
3,"$281,176,332.44 $33,004,446.68 11.74% $32,324,...",DraftKings
4,"$59,682,063.71 $6,599,625.36 11.06% $6,453,759...",Fanatics
5,"$150,594,846.88 $17,165,339.33 11.40% $16,801,...",FanDuel
6,"$21,981,883.07 $1,676,874.69 7.63% $1,622,011....",theScore Bet
7,"$587,101,900.84 $66,530,607.93 11.33% $65,101,...",Total


In [4]:
print("Category 3 operators:", sorted(category3["operator"].tolist()))
category3

Category 3 operators: ['Bally Bet', 'BetMGM', 'Caesars Sportsbook', 'DraftKings', 'FanDuel', 'Fanatics', 'theScore Bet']


,operator,period_start,wagers_settled,taxable_revenue,tax_collected
0,Bally Bet,2026-07-01,3.272046e+06,383965.58,76793.12
1,BetMGM,2026-07-01,5.279115e+07,6386566.27,1277313.25
2,Caesars Sportsbook,2026-07-01,1.760358e+07,1129965.58,225993.01
3,DraftKings,2026-07-01,2.811763e+08,32324402.68,6464880.54
4,Fanatics,2026-07-01,5.968206e+07,6453759.09,1290751.82
5,FanDuel,2026-07-01,1.505948e+08,16801318.96,3360263.79
6,theScore Bet,2026-07-01,2.198188e+07,1622011.39,324402.28


In [5]:
parsed_totals = category3[["wagers_settled", "taxable_revenue", "tax_collected"]].sum()
reconciliation = pd.DataFrame(
    {
        "metric": ["wagers_settled", "taxable_revenue", "tax_collected"],
        "operator_sum": [
            parsed_totals["wagers_settled"],
            parsed_totals["taxable_revenue"],
            parsed_totals["tax_collected"],
        ],
        "total_online_row": [
            total_online["wagers_settled"],
            total_online["taxable_revenue"],
            total_online["tax_collected"],
        ],
    }
)
reconciliation["matches"] = [
    round(operator_sum, 2) == round(official_total, 2)
    for operator_sum, official_total in zip(
        reconciliation["operator_sum"], reconciliation["total_online_row"]
    )
]
assert reconciliation["matches"].all(), reconciliation
reconciliation

,metric,operator_sum,total_online_row,matches
0,wagers_settled,5.871019e+08,5.871019e+08,True
1,taxable_revenue,6.510199e+07,6.510199e+07,True
2,tax_collected,1.302040e+07,1.302040e+07,True


## Why this is *taxable revenue*, not GGR

Massachusetts sports-wagering PDFs label the revenue column **Taxable Gaming Revenue**. Under MGL c. 23N
that figure is accrual win minus patron winnings and federal excise tax — a tax base defined in law.

It is **not** the same as gross gaming revenue (GGR) used in many other states. Silently renaming
`taxable_revenue` to `gross_revenue` would misstate the metric and break cross-state comparisons.

## Logic to promote later into `massachusetts.py`

When a collector is added, these notebook steps map cleanly to functions:

1. `discover_consolidated_report_url(html)` — pick the monthly `MGC-Revenue-Report-*.pdf`.
2. `save_raw_bytes(..., state_code="MA")` — immutable raw archive with SHA-256 prefix.
3. `find_online_operator_page(pdf)` — skip MoM/YoY/DFS/chart pages; keep the single-month ONLINE LICENSEE page.
4. `parse_online_section(text, period_start)` — emit operator rows + capture `Total Online` for reconciliation.
5. Map columns explicitly: `wagers_settled`, `taxable_revenue`, `tax_collected` (not GGR).
6. Exclude Category 1 retail operators and total rows before upsert.

Negative values flow through unchanged via `parse_money()` — do not strip leading minus signs.